# Exercise 04 — Diversification

MSc Finance · Investments · FHNW · Autumn 2026

Lecture 04 made two claims about diversification and proved them for two assets: risk
falls whenever $\rho < 1$, and the fall stops at a floor set by the average covariance.
This exercise runs both claims on twenty stocks, and then builds an artificial market in
which the floor is known in advance, so you can see where it comes from rather than
inferring it from a curve.

**How to work with this notebook.** The task text is here and on the exercise sheet. Each
code cell is a stub: the `# TODO` lines are the steps, in order. The setup and data cells
below are complete — run them first and leave them alone.

Run **Runtime → Restart and run all** before you trust any number in here. Tasks 2 and 3
draw random numbers, so a cell run on its own picks up wherever the generator happens to
be; a clean top-to-bottom run gives everyone in the room the same figures.

**Data.** Twenty SPI constituents, monthly total-return index levels in Swiss francs,
December 2004 to December 2025 — 253 month-ends and 252 monthly returns. Prices are from
S&P Capital IQ (dividend- and split-adjusted); the `RiskFree` sheet holds the SARON from
the [Swiss National Bank data portal](https://data.snb.ch) and is not needed this week.

Three caveats sit in the `Info` sheet and one of them matters for every number you are
about to compute. The twenty names are stocks that are *still* in the SPI and have traded
continuously since 2004, so anything that was delisted, acquired or failed is missing and
the returns are biased upward. There is deliberately no SPI series in the file: the
equally weighted portfolio of the twenty is the market proxy, so that the proxy and its
constituents carry the same selection bias.

In [ ]:
!wget -q https://raw.githubusercontent.com/KroeTiA/Investments/main/exercise_utils.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

from exercise_utils import FHNW, ASSET_CYCLE, setup_style, load_returns, save_results
setup_style()

SEED = 42
rng = np.random.default_rng(SEED)

In [ ]:
BASE = "https://raw.githubusercontent.com/KroeTiA/Investments/main/"
DATA_URL = BASE + "Exercise_04/data/swiss_equities_monthly.xlsx"

# Monthly total-return index levels -> monthly returns.
rets = load_returns(DATA_URL, sheet="Prices", index_col="Date", to_returns=True)

# The Info sheet is a two-column field/value list; the ticker rows are the tail of it.
info = pd.read_excel(DATA_URL, sheet_name="Info")
tick = info[info["Field"].astype(str).str.startswith("SWX:")]
NAME = dict(zip(tick["Field"], tick["Value"]))

N_ASSETS = rets.shape[1]
print(f"{rets.shape[0]} monthly returns, {N_ASSETS} stocks, "
      f"{rets.index.min():%b %Y} to {rets.index.max():%b %Y}")
print(f"{N_ASSETS * (N_ASSETS - 1) // 2} distinct pairs")

## Task 1 — Twenty stocks instead of two

Slide 20 put the volatility of a two-asset portfolio next to the weighted average of the
two individual volatilities and called the gap the diversification benefit: 16 % against
13.11 %, so 2.89 percentage points. Do the same with twenty stocks.

With equal weights $w_i = 1/N$ the weighted average of the individual volatilities is
simply their mean $\bar{\sigma}$, so the comparison is between $\bar{\sigma}$ and

$$\sigma_P = \sqrt{\mathbf{w}' \Sigma\, \mathbf{w}}, \qquad \mathbf{w} = (1/N, \dots, 1/N)'.$$

The returns are monthly; annualise volatilities with $\sqrt{12}$ and the covariance matrix
with 12. Compute four numbers: the average pairwise correlation $\bar{\rho}$ over the 190
distinct pairs, the average annualised volatility $\bar{\sigma}$ of a single stock,
$\sigma_P$ for the equally weighted portfolio of all twenty, and the systematic floor
$\bar{\sigma}\sqrt{\bar{\rho}}$. Package them in a function `risk_summary(r)` — task 4
reuses it on subsamples.

Report the diversification benefit $\bar{\sigma} - \sigma_P$ in percentage points against
the 2.89 pp of the two-asset example, and identify the most and the least correlated pair
in the sample by name.

*Deliverable: the four numbers, the benefit in percentage points, and the two extreme
pairs.*

In [ ]:
ANNUAL = np.sqrt(12)
def risk_summary(r):
    n = r.shape[1]
    upper = np.triu(np.ones((n, n), dtype=bool), 1)
    # TODO: rho_bar, the mean of the n(n-1)/2 distinct pairwise correlations
    # TODO: sd_bar, the mean annualised volatility of a single stock
    # TODO: sigma_P, the annualised volatility of the 1/N portfolio, sqrt(w' Sigma w)
    # TODO: return the three, plus the systematic floor sd_bar * sqrt(rho_bar)
s = risk_summary(rets)
print(s.to_string(float_format="{:.4f}".format))
print(f"\ndiversification benefit  sd_bar - sigma_P = "
      f"{100 * (s['sd_bar'] - s['sigma_P']):.2f} pp")

In [ ]:
upper = np.triu(np.ones((N_ASSETS, N_ASSETS), dtype=bool), 1)
# TODO: the 190 pairwise correlations as one series, then its range and its extremes
print(f"pairwise correlations from {pairs.min():.2f} to {pairs.max():.2f}")
for label, key in [("lowest", pairs.idxmin()), ("highest", pairs.idxmax())]:
    print(f"  {label:8s} {pairs[key]:.2f}   {NAME[key[0]]} / {NAME[key[1]]}")

## Task 2 — The diversification curve

Slide 8 describes an experiment: draw $N$ stocks at random, form the equally weighted
portfolio, record its volatility, repeat. Slide 9 reports what Statman found running it on
U.S. data — 49 % for a single stock falling to about 19–20 % and flattening there. Run the
experiment on the Swiss sample.

For every $N$ from 1 to 20, draw 500 random subsets of size $N$ without replacement with
`rng.choice(n, size=N, replace=False)`, compute the annualised volatility of the equally
weighted portfolio of each subset, and average over the 500. Plot the resulting curve
against $N$ and mark the floor from task 1.

Then read three things off it. First, $\sigma$ at $N = 1, 5, 10$ and $20$. Second, the
share of the risk reduction available within twenty stocks that the first five deliver,
$(\sigma_1 - \sigma_5)/(\sigma_1 - \sigma_{20})$. Third, the same share measured against
the floor instead of against $N = 20$. The two shares are not the same number, and the
gap between them is the point.

*Deliverable: one figure, the four volatilities, and the two shares.*

In [ ]:
# TODO: annualised volatility of the equally weighted portfolio of `cols`
def portfolio_sd(cov, cols):
    ...
def diversification_curve(cov, draws=500):
    n = cov.shape[0]
    # TODO: for each N from 1 to n, draw `draws` random subsets of size N without
    # TODO: replacement and average portfolio_sd over them
    return pd.Series(out, name="sigma")
COV = rets.cov().values * 12
curve = diversification_curve(COV)
print(curve.loc[[1, 5, 10, 20]].to_string(float_format="{:.4f}".format))

In [ ]:
fig1, ax = plt.subplots(figsize=(8.2, 4.6))
# TODO: the curve, and the systematic floor as a horizontal reference line
ax.set_xlabel("Number of stocks N")
ax.set_ylabel("Annualised volatility")
ax.set_xticks(range(0, 21, 5))
ax.set_ylim(0.10, 0.25)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.legend()
plt.show()
# TODO: the share of the achievable reduction that the first five stocks deliver,
# TODO: measured once against N = 20 and once against the floor

## Task 3 — Where the floor comes from

The curve in task 2 flattens above a floor it never reaches, and slide 33 says what sets
that floor: the average covariance, which is $\bar{\rho}\,\bar{\sigma}^2$ when every stock
has the same volatility. Rather than infer that from the data, build a market in which it
is true by construction and watch the same curve come out.

Start from $\bar{\sigma}$ and $\bar{\rho}$ of task 1 and write down the equicorrelation
covariance matrix of twenty assets: volatility $\bar{\sigma}$ for every asset, correlation
$\bar{\rho}$ for every pair. Simulate 252 monthly returns from it with
`rng.multivariate_normal` — the matrix has to be in monthly units, so divide the annual
one by 12 — estimate the sample covariance matrix of the simulated returns, and run
`diversification_curve` on it. Plot it on top of the empirical curve. Twenty real Swiss
stocks and twenty artificial assets that share nothing but two numbers should land close
to each other; report how close, and run `risk_summary` on the simulated returns as well.
You put $\bar{\sigma}$ and $\bar{\rho}$ into the simulation and 252 months came back out:
the second call tells you what the sample gives you back, and the difference between the
two curves is not a modelling failure but that.

Then vary the correlation. Hold $\bar{\sigma}$ fixed and repeat for
$\rho \in \{0,\ 0.15,\ 0.35,\ 0.6\}$, drawing the four curves and their four floors
$\bar{\sigma}\sqrt{\rho}$ in one figure.

Finally answer the question the figure poses. For an equicorrelation market the curve has
a closed form,

$$\sigma_P(N) = \bar{\sigma}\,\sqrt{\rho + \frac{1-\rho}{N}},$$

which you can invert. For each of the four correlations, how many stocks are needed to
come within one percentage point of the floor? Report the four values of $N$ next to the
four floors, and be ready to say which of the two columns an investor should care about.

*Deliverable: the overlay figure, the four-curve figure, and a table of four floors and
four values of N.*

In [ ]:
# TODO: annualised covariance matrix -- rho off the diagonal, 1 on it, scaled by sd^2
def equicorrelation_cov(sd, rho, n=20):
    ...
# TODO: T monthly draws from a zero-mean normal with cov_annual / 12
def simulate(cov_annual, T=252):
    ...
SD_BAR, RHO_BAR = s["sd_bar"], s["rho_bar"]
# TODO: one simulated market at (SD_BAR, RHO_BAR), then the same curve on its sample cov
fig2, ax = plt.subplots(figsize=(8.2, 4.6))
ax.plot(curve.index, curve.values, marker="o", ms=4, lw=1.6,
        color=FHNW["blue"], label="20 Swiss stocks")
ax.plot(curve_sim.index, curve_sim.values, marker="s", ms=4, lw=1.6,
        color=FHNW["orange"], label="simulated equicorrelation market")
ax.axhline(s["floor"], color=FHNW["red"], ls="--", lw=1.3,
           label=f"systematic floor {s['floor']:.1%}")
ax.set_xlabel("Number of stocks N")
ax.set_ylabel("Annualised volatility")
ax.set_xticks(range(0, 21, 5))
ax.set_ylim(0.10, 0.25)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.legend()
plt.show()
# TODO: the largest gap between the two curves, and what the simulated sample
# TODO: gives back for the two parameters that went into it

In [ ]:
RHOS = [0.0, 0.15, 0.35, 0.6]
rows = []
fig3, ax = plt.subplots(figsize=(8.2, 4.6))
for rho, colour in zip(RHOS, ASSET_CYCLE):
    ...
    # TODO: simulate a market with this rho, run the curve, draw it and its floor
    # TODO: invert sigma_P(N) = sd_bar * sqrt(rho + (1 - rho) / N) at floor + 1 pp
    rows.append({"rho": rho, "floor": floor, "sigma(N=20)": c[20],
                 "N within 1 pp": np.ceil(n_within)})
ax.set_xlabel("Number of stocks N")
ax.set_ylabel("Annualised volatility")
ax.set_xticks(range(0, 21, 5))
ax.set_ylim(0, 0.26)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.legend(ncol=2)
plt.show()
print(pd.DataFrame(rows).to_string(index=False,
                                   float_format="{:.4f}".format))

## Task 4 (optional) — The floor in a crisis

Slide 35 ends with a warning: correlations rise in a downturn, and diversification fails
exactly when it is needed. Test it, and then check what the warning is worth.

Estimate the four numbers of task 1 separately on three windows — the calm run-up
(2005–2007), the crisis (2008–2009), and everything after (2010–2025). `risk_summary` does
the work and the slicing is one line each; note how short the first two windows are, 36
and 24 monthly observations for a matrix with 190 distinct correlations in it.

Slide 35 blames the correlation. But the floor is a product,
$\bar{\sigma}\sqrt{\bar{\rho}}$, and a product has two factors. Decompose the change in
the floor from the calm window to the crisis: once holding $\bar{\rho}$ at its calm value
and letting $\bar{\sigma}$ move to its crisis value, once the other way round. One factor
carries most of the change.

*Deliverable: a three-row table, the two counterfactual floors, and one sentence naming
the factor that does the work.*

In [ ]:
WINDOWS = {"2005-2007 calm":   ("2005", "2007"),
           "2008-2009 crisis": ("2008", "2009"),
           "2010-2025 after":  ("2010", "2025")}
# TODO: risk_summary on each window, collected into one table
calm, crisis = sub.loc["2005-2007 calm"], sub.loc["2008-2009 crisis"]
# TODO: the two counterfactual floors -- move one factor at a time

## Export

Bundle the figures and the tables, in case you want them for the transfer questions or
your own notes.

In [ ]:
# TODO: export the curve and the summary tables together with the figures you want

## Where this goes next

Two quizzes are open in Moodle until Sunday: the cumulative drill and the transfer
questions. Both are ungraded, and both are exactly the format the exams use.

Every portfolio in this notebook held equal weights. That was a choice, and not obviously
a good one — Nestlé at 13.6 % volatility and UBS at 31.9 % contributed the same share of
capital and very different shares of risk. Lecture 05 drops the constraint and asks which
weights are best, which turns the single curve you plotted into a frontier of them.